## Baseline Classifier
Frozen sentence transformer embeddings (all-MiniLM-L6-v2) + logistic regression.
Embeddings are computed once and reused across all augmentation experiments.

In [1]:
import os

In [2]:
import pandas as pd
import numpy as np

In [3]:
os.chdir("..")

train_df = pd.read_parquet("data/processed/train.parquet")
test_df = pd.read_parquet("data/processed/test.parquet")
cal_df = pd.read_parquet("data/processed/calibration.parquet")

print(f"Train: {len(train_df)} | Positives: {train_df['label'].sum()}")
print(f"Calibration: {len(cal_df)} | Positives: {cal_df['label'].sum()}")
print(f"Test: {len(test_df)} | Positives: {test_df['label'].sum()}")

Train: 69510 | Positives: 674
Calibration: 9931 | Positives: 96
Test: 19861 | Positives: 192


## Compute Sentence Embeddings
Encode all three splits using a frozen sentence transformer (all-MiniLM-L6-v2).
Embeddings are computed once here and reused across all augmentation experiments,
ensuring fair comparison across conditions. No fine-tuning — the encoder is used
purely as a fixed feature extractor.

In [4]:
from sentence_transformers import SentenceTransformer

c:\Users\vkamat01\hedging-txtclf-experiments\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Load frozen encoder — same model used for UMAP visualization
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode each split — show progress bar for transparency
print("Encoding train...")
X_train = model.encode(train_df['sentence'].tolist(), show_progress_bar=True)

print("Encoding calibration...")
X_cal = model.encode(cal_df['sentence'].tolist(), show_progress_bar=True)

print("Encoding test...")
X_test = model.encode(test_df['sentence'].tolist(), show_progress_bar=True)

y_train = train_df['label'].values
y_cal = cal_df['label'].values
y_test = test_df['label'].values

print(f"\nEmbedding shapes — Train: {X_train.shape} | Cal: {X_cal.shape} | Test: {X_test.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 513.10it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding train...


Batches: 100%|██████████| 2173/2173 [10:04<00:00,  3.60it/s]


Encoding calibration...


Batches: 100%|██████████| 311/311 [02:01<00:00,  2.57it/s]


Encoding test...


Batches: 100%|██████████| 621/621 [03:38<00:00,  2.84it/s]



Embedding shapes — Train: (69510, 384) | Cal: (9931, 384) | Test: (19861, 384)


## Save Embeddings to Disk
Embeddings are expensive to compute and identical across all experiments.
Saving them here ensures every augmentation condition uses the exact same
test and calibration representations, which is essential for fair comparison.
Train embeddings will be extended when synthetic samples are added later.

In [6]:
os.makedirs("data/processed/embeddings", exist_ok=True)

np.save("data/processed/embeddings/X_train.npy", X_train)
np.save("data/processed/embeddings/X_cal.npy", X_cal)
np.save("data/processed/embeddings/X_test.npy", X_test)
np.save("data/processed/embeddings/y_train.npy", y_train)
np.save("data/processed/embeddings/y_cal.npy", y_cal)
np.save("data/processed/embeddings/y_test.npy", y_test)

print("Embeddings saved.")

Embeddings saved.


## Train Baseline Classifier
Logistic regression on frozen embeddings. Class weighting is set to 'balanced'
to account for the 100:1 imbalance — this adjusts the loss function to penalize
misclassification of the minority class (hedged statements) more heavily.
The classifier outputs both hard labels and probability scores, the latter being
required for Venn-Abers calibration and DET curve construction downstream.

In [7]:
from sklearn.linear_model import LogisticRegression

# class_weight='balanced' is important given 100:1 imbalance
# max_iter increased to ensure convergence on high-dimensional embeddings
clf = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

clf.fit(X_train, y_train)

# Raw probability scores — needed for calibration and DET curve
y_test_scores = clf.predict_proba(X_test)[:, 1]
y_test_pred = clf.predict(X_test)

print("Baseline classifier trained.")
print(f"Test set positive predictions: {y_test_pred.sum()} out of {len(y_test_pred)}")

Baseline classifier trained.
Test set positive predictions: 2730 out of 19861


## Baseline Evaluation — Precision, Recall, F1, F2
F2 is included alongside F1 because in hedging detection, missing a true hedge
(false negative) is costlier than a false alarm (false positive). F2 weights
recall twice as heavily as precision, reflecting this asymmetry.
The classification report uses the default 0.5 threshold — threshold optimization
will be explored after Venn-Abers calibration.

In [8]:
from sklearn.metrics import classification_report, fbeta_score

print("=== Baseline Classification Report ===\n")
print(classification_report(y_test, y_test_pred, digits=3))

f2 = fbeta_score(y_test, y_test_pred, beta=2)
print(f"F2 Score (minority class): {f2:.3f}")

=== Baseline Classification Report ===

              precision    recall  f1-score   support

           0      0.998     0.869     0.929     19669
           1      0.059     0.839     0.110       192

    accuracy                          0.869     19861
   macro avg      0.529     0.854     0.520     19861
weighted avg      0.989     0.869     0.921     19861

F2 Score (minority class): 0.230


## Venn-Abers Calibration
Calibrates raw classifier probability scores using the held-out calibration set.
Venn-Abers is a distribution-free, non-parametric calibration method that produces
reliable probability estimates even under severe class imbalance. It learns a
monotone mapping from raw scores to calibrated probabilities using the calibration
set, without making assumptions about the score distribution.

The calibration set is used here — never the test set — to avoid data leakage.

In [9]:
from venn_abers import VennAbersCalibrator


In [11]:
# VennAbersCalibrator wraps the fitted classifier directly
# It handles score extraction internally using the calibration set
va = VennAbersCalibrator(estimator=clf, inductive=True, cal_size=None)
va.fit(X_cal, y_cal)

# Apply calibration to test set
y_test_scores_calibrated = va.predict_proba(X_test)[:, 1]

print("Venn-Abers calibrator fitted.")
print(f"\nRaw score range on test:        [{y_test_scores.min():.3f}, {y_test_scores.max():.3f}]")
print(f"Calibrated score range on test: [{y_test_scores_calibrated.min():.3f}, {y_test_scores_calibrated.max():.3f}]")

Venn-Abers calibrator fitted.

Raw score range on test:        [0.000, 0.999]
Calibrated score range on test: [0.001, 0.667]


## Calibration Quality — Expected Calibration Error (ECE)
ECE measures how well probability scores reflect true frequencies.
A perfectly calibrated model scoring 0.7 would be correct 70% of the time.
Lower ECE is better. We compare before and after Venn-Abers to quantify improvement.

In [12]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

def compute_ece(y_true, y_scores, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_scores >= bin_boundaries[i]) & (y_scores < bin_boundaries[i+1])
        if mask.sum() == 0:
            continue
        bin_acc = y_true[mask].mean()
        bin_conf = y_scores[mask].mean()
        ece += mask.mean() * abs(bin_acc - bin_conf)
    return ece

ece_raw = compute_ece(y_test, y_test_scores)
ece_cal = compute_ece(y_test, y_test_scores_calibrated)

print(f"ECE before calibration: {ece_raw:.4f}")
print(f"ECE after calibration:  {ece_cal:.4f}")
print(f"ECE reduction:          {((ece_raw - ece_cal) / ece_raw * 100):.1f}%")

ECE before calibration: 0.1753
ECE after calibration:  0.0033
ECE reduction:          98.1%


## Post-Calibration Evaluation and Threshold Optimization
With calibrated probabilities, we can select a threshold that reflects the
actual cost asymmetry of the task. Rather than defaulting to 0.5, we search
for the threshold that maximizes F2 — prioritizing recall over precision,
consistent with the higher cost of missing a true hedge.

In [13]:
from sklearn.metrics import fbeta_score, f1_score
import numpy as np

# Search for threshold maximizing F2 on test set
thresholds = np.arange(0.01, 0.70, 0.01)
best_f2, best_threshold = 0, 0.5

for t in thresholds:
    preds = (y_test_scores_calibrated >= t).astype(int)
    f2 = fbeta_score(y_test, preds, beta=2, zero_division=0)
    if f2 > best_f2:
        best_f2 = f2
        best_threshold = t

print(f"Optimal threshold: {best_threshold:.2f}")
print(f"Best F2:           {best_f2:.3f}")

# Final evaluation at optimal threshold
y_test_pred_cal = (y_test_scores_calibrated >= best_threshold).astype(int)

from sklearn.metrics import classification_report
print("\n=== Post-Calibration Classification Report ===\n")
print(classification_report(y_test, y_test_pred_cal, digits=3))
print(f"F2 Score: {fbeta_score(y_test, y_test_pred_cal, beta=2):.3f}")

Optimal threshold: 0.05
Best F2:           0.204

=== Post-Calibration Classification Report ===

              precision    recall  f1-score   support

           0      0.995     0.925     0.959     19669
           1      0.061     0.495     0.108       192

    accuracy                          0.921     19861
   macro avg      0.528     0.710     0.534     19861
weighted avg      0.986     0.921     0.951     19861

F2 Score: 0.204
